# HIGGS Studio · free Kaggle GPU backend

This notebook turns a free Kaggle GPU into the generation engine for your HIGGS Studio on your Mac.

**One-time setup**
1. Kaggle account with a verified phone number (needed for GPU + internet).
2. In the right sidebar: **Session options → Accelerator → GPU T4 x2**, and **Internet → On**.

**Every session**
1. Click **Run All**.
2. Wait for the last cell to print an address like `https://something.trycloudflare.com` and an access token.
3. In HIGGS Studio → **Settings → Kaggle free GPU**, paste both, click **Test**, then **Save**.
4. Generate. Leave this tab open; stop the session when you're done to save GPU hours.

**Limits (Kaggle's, not ours):** about 30 GPU hours per week, sessions end after at most 12 hours, and an idle session may be stopped. Model files (~30 GB) are re-downloaded each session, which takes a few minutes on Kaggle's network.

## 1 · Settings

In [ ]:
# Which engines to download this session (True/False)
INSTALL_WAN22_5B = True     # Wan 2.2 TI2V 5B: best quality you can run here (~18 GB)
INSTALL_LTXV_2B  = True     # LTX-Video 2B distilled: fast drafts (~11.5 GB)
INSTALL_UPSCALER = True     # Real-ESRGAN x2 for AI upscaling (70 MB)

COMFY_DIR  = "/kaggle/temp/ComfyUI" if __import__("os").path.isdir("/kaggle/temp") else "/tmp/ComfyUI"

## 2 · Check the GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU. Set Session options -> Accelerator -> GPU T4 x2 and restart."
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))

## 3 · Install ComfyUI (keeps Kaggle's CUDA PyTorch)

In [ ]:
import os, subprocess, sys
if not os.path.isdir(COMFY_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", COMFY_DIR], check=True)
req = [l for l in open(f"{COMFY_DIR}/requirements.txt").read().splitlines()
       if l.strip() and not l.startswith("#") and not l.split("=")[0].split(">")[0].split("<")[0].strip() in ("torch", "torchvision", "torchaudio")]
open("/tmp/comfy_req.txt", "w").write("\n".join(req))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/comfy_req.txt", "hf_transfer", "huggingface_hub"], check=True)
print("ComfyUI ready at", COMFY_DIR)

## 4 · Install the HIGGS bridge (model manager + access token)

In [ ]:
bridge_src = "\"\"\"HIGGS Studio bridge for ComfyUI.\n\nAdds a few HTTP routes to ComfyUI so the studio can manage models on any backend\n(this computer or the Kaggle notebook) and see hardware usage:\n\n  GET  /higgs/info        hardware, VRAM/RAM usage, free disk, versions\n  POST /higgs/download    {url, folder, filename}  -> downloads into ComfyUI/models/<folder>\n  GET  /higgs/downloads   progress of downloads\n  POST /higgs/delete      {folder, filename}\n\nIf the environment variable HIGGS_TOKEN is set, every ComfyUI request must carry\n\"Authorization: Bearer <token>\" (or ?token=<token>). The Kaggle notebook sets this\nso a public tunnel URL can't be used by anyone else.\n\nAdds no graph nodes.\n\"\"\"\nfrom __future__ import annotations\n\nimport os\nimport shutil\nimport threading\nimport time\nimport urllib.request\nfrom pathlib import Path\n\nimport folder_paths\nfrom aiohttp import web\nfrom server import PromptServer\n\nNODE_CLASS_MAPPINGS: dict = {}\nNODE_DISPLAY_NAME_MAPPINGS: dict = {}\nWEB_DIRECTORY = None\n\nALLOWED_FOLDERS = {\"diffusion_models\", \"text_encoders\", \"vae\", \"checkpoints\", \"upscale_models\", \"loras\", \"clip_vision\"}\nALLOWED_HOSTS = (\"https://huggingface.co/\", \"https://github.com/\")\n_downloads: dict[str, dict] = {}\n_lock = threading.Lock()\n\nroutes = PromptServer.instance.routes\nTOKEN = os.environ.get(\"HIGGS_TOKEN\", \"\").strip()\n\n\n# ------------------------------------------------------------------ auth\nif TOKEN:\n    @web.middleware\n    async def _higgs_auth(request: web.Request, handler):\n        if request.method == \"OPTIONS\":\n            return await handler(request)\n        auth = request.headers.get(\"Authorization\", \"\")\n        if auth == f\"Bearer {TOKEN}\" or request.query.get(\"token\") == TOKEN:\n            return await handler(request)\n        return web.json_response({\"error\": \"unauthorized\"}, status=401)\n\n    try:\n        PromptServer.instance.app.middlewares.insert(0, _higgs_auth)\n        print(\"[higgs_bridge] token auth enabled\")\n    except RuntimeError as e:  # app already frozen (very old/new ComfyUI)\n        print(f\"[higgs_bridge] WARNING could not enable token auth: {e}\")\n\n\n# ------------------------------------------------------------------ helpers\ndef _folder_dir(folder: str) -> Path:\n    if folder not in ALLOWED_FOLDERS:\n        raise ValueError(f\"folder '{folder}' not allowed\")\n    paths = folder_paths.get_folder_paths(folder)\n    if not paths:\n        raise ValueError(f\"ComfyUI has no path for '{folder}'\")\n    p = Path(paths[0])\n    p.mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef _download(key: str, url: str, dest: Path) -> None:\n    part = dest.with_suffix(dest.suffix + \".part\")\n    headers = {\"User-Agent\": \"higgs-studio/1.0\"}\n    hf_token = os.environ.get(\"HF_TOKEN\")\n    if hf_token and url.startswith(\"https://huggingface.co/\"):\n        headers[\"Authorization\"] = f\"Bearer {hf_token}\"\n    try:\n        req = urllib.request.Request(url, headers=headers)\n        with urllib.request.urlopen(req, timeout=60) as r, part.open(\"wb\") as fh:\n            total = int(r.headers.get(\"Content-Length\") or 0)\n            with _lock:\n                _downloads[key][\"total\"] = total\n            done, t0 = 0, time.time()\n            while True:\n                chunk = r.read(1 << 22)\n                if not chunk:\n                    break\n                fh.write(chunk)\n                done += len(chunk)\n                with _lock:\n                    d = _downloads[key]\n                    d[\"done\"] = done\n                    d[\"speed_mb_s\"] = round(done / max(time.time() - t0, 0.01) / 2**20, 1)\n                    if d.get(\"cancel\"):\n                        raise RuntimeError(\"cancelled\")\n        part.replace(dest)\n        with _lock:\n            _downloads[key][\"status\"] = \"done\"\n    except Exception as e:  # noqa: BLE001\n        part.unlink(missing_ok=True)\n        with _lock:\n            _downloads[key][\"status\"] = \"error\"\n            _downloads[key][\"error\"] = str(e)\n\n\n# ------------------------------------------------------------------ routes\n@routes.get(\"/higgs/info\")\nasync def higgs_info(request):\n    info: dict = {\"bridge_version\": 1, \"auth\": bool(TOKEN)}\n    try:\n        import torch\n        info[\"torch\"] = torch.__version__\n        if torch.cuda.is_available():\n            i = torch.cuda.current_device()\n            free, total = torch.cuda.mem_get_info(i)\n            info[\"gpu\"] = {\"name\": torch.cuda.get_device_name(i), \"vram_total_gb\": round(total / 2**30, 1),\n                           \"vram_used_gb\": round((total - free) / 2**30, 1), \"count\": torch.cuda.device_count()}\n        elif getattr(torch.backends, \"mps\", None) and torch.backends.mps.is_available():\n            info[\"gpu\"] = {\"name\": \"Apple Metal (MPS)\"}\n        else:\n            info[\"gpu\"] = None\n    except Exception as e:  # noqa: BLE001\n        info[\"torch_error\"] = str(e)\n    try:\n        import psutil\n        vm = psutil.virtual_memory()\n        info[\"ram_total_gb\"] = round(vm.total / 2**30, 1)\n        info[\"ram_used_gb\"] = round((vm.total - vm.available) / 2**30, 1)\n    except Exception:  # noqa: BLE001\n        pass\n    models_dir = Path(folder_paths.models_dir)\n    du = shutil.disk_usage(models_dir)\n    info[\"disk_free_gb\"] = round(du.free / 2**30, 1)\n    info[\"models_dir\"] = str(models_dir)\n    return web.json_response(info)\n\n\n@routes.post(\"/higgs/download\")\nasync def higgs_download(request):\n    body = await request.json()\n    url, folder, filename = body.get(\"url\", \"\"), body.get(\"folder\", \"\"), Path(body.get(\"filename\", \"\")).name\n    if not url.startswith(ALLOWED_HOSTS) or not filename:\n        return web.json_response({\"error\": \"only Hugging Face / GitHub downloads are allowed\"}, status=400)\n    try:\n        dest = _folder_dir(folder) / filename\n    except ValueError as e:\n        return web.json_response({\"error\": str(e)}, status=400)\n    key = f\"{folder}/{filename}\"\n    with _lock:\n        cur = _downloads.get(key)\n        if cur and cur[\"status\"] == \"running\":\n            return web.json_response({\"key\": key, \"status\": \"running\"})\n        if dest.exists():\n            _downloads[key] = {\"key\": key, \"status\": \"done\", \"done\": dest.stat().st_size, \"total\": dest.stat().st_size}\n            return web.json_response({\"key\": key, \"status\": \"done\"})\n        _downloads[key] = {\"key\": key, \"status\": \"running\", \"done\": 0, \"total\": 0, \"url\": url, \"started\": time.time()}\n    threading.Thread(target=_download, args=(key, url, dest), daemon=True).start()\n    return web.json_response({\"key\": key, \"status\": \"running\"})\n\n\n@routes.get(\"/higgs/downloads\")\nasync def higgs_downloads(request):\n    with _lock:\n        return web.json_response({\"downloads\": list(_downloads.values())})\n\n\n@routes.post(\"/higgs/delete\")\nasync def higgs_delete(request):\n    body = await request.json()\n    try:\n        path = _folder_dir(body.get(\"folder\", \"\")) / Path(body.get(\"filename\", \"\")).name\n    except ValueError as e:\n        return web.json_response({\"error\": str(e)}, status=400)\n    if path.exists():\n        path.unlink()\n        return web.json_response({\"deleted\": True})\n    return web.json_response({\"deleted\": False})\n"
os.makedirs(f"{COMFY_DIR}/custom_nodes/higgs_bridge", exist_ok=True)
open(f"{COMFY_DIR}/custom_nodes/higgs_bridge/__init__.py", "w").write(bridge_src)
print("bridge installed")

## 5 · Download models (free, from Hugging Face)

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download
M = f"{COMFY_DIR}/models"
def get(repo, path, folder):
    dest = f"{M}/{folder}"
    os.makedirs(dest, exist_ok=True)
    name = path.split("/")[-1]
    if os.path.exists(f"{dest}/{name}"):
        print("have", name); return
    p = hf_hub_download(repo, path, local_dir="/tmp/hf_dl")
    os.replace(p, f"{dest}/{name}")
    print("downloaded", name)

if INSTALL_WAN22_5B:
    get("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors", "diffusion_models")
    get("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/vae/wan2.2_vae.safetensors", "vae")
    get("Comfy-Org/Wan_2.1_ComfyUI_repackaged", "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", "text_encoders")
if INSTALL_LTXV_2B:
    get("Lightricks/LTX-Video", "ltxv-2b-0.9.8-distilled.safetensors", "checkpoints")
    get("comfyanonymous/flux_text_encoders", "t5xxl_fp8_e4m3fn_scaled.safetensors", "text_encoders")
if INSTALL_UPSCALER:
    get("ai-forever/Real-ESRGAN", "RealESRGAN_x2.pth", "upscale_models")

## 6 · Start ComfyUI + a free Cloudflare tunnel
The access token stops anyone else who sees the address from using your GPU.

In [ ]:
import os, re, secrets, subprocess, time, urllib.request
TOKEN = secrets.token_urlsafe(18)
env = dict(os.environ, HIGGS_TOKEN=TOKEN)
log = open("/tmp/comfy.log", "w")
comfy = subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--preview-method", "none"],
                         cwd=COMFY_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/system_stats", headers={"Authorization": f"Bearer {TOKEN}"}), timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    print(open("/tmp/comfy.log").read()[-3000:]); raise SystemExit("ComfyUI did not start")
print("ComfyUI is running")

if not os.path.exists("/tmp/cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/tmp/cloudflared")
    os.chmod("/tmp/cloudflared", 0o755)
tlog = open("/tmp/tunnel.log", "w")
tunnel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8188"], stdout=tlog, stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/tunnel.log").read())
    if m:
        URL = m.group(0); break
    time.sleep(1)
print("\n" + "=" * 70)
print("  Paste these into HIGGS Studio -> Settings -> Kaggle free GPU")
print("  Address:", URL)
print("  Token:  ", TOKEN)
print("=" * 70)

## 7 · Keep alive
Leave this running while you generate. Stop it (■) and end the session when you're done.

In [ ]:
import time, json, urllib.request
while True:
    try:
        r = urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:8188/queue", headers={"Authorization": f"Bearer {TOKEN}"}), timeout=5)
        q = json.loads(r.read())
        print(time.strftime("%H:%M:%S"), "running", len(q["queue_running"]), "waiting", len(q["queue_pending"]), "| address", URL, flush=True)
    except Exception as e:
        print("ComfyUI not responding:", e); print(open("/tmp/comfy.log").read()[-2000:])
    if comfy.poll() is not None:
        print(open("/tmp/comfy.log").read()[-3000:]); break
    time.sleep(60)